In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)
from collections import Counter

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

# 1. Load dataset
df = pd.read_csv("../Data/sentiment_data.csv")

# Keep only useful columns
df = df[["Comment", "Sentiment"]]

# Remove missing or empty comments
df = df.dropna(subset=["Comment", "Sentiment"])
df = df[df["Comment"].str.strip() != ""] # .str.strip()：移除字串開頭與結尾的空白。
df = df.reset_index(drop=True) # 當我們刪除資料後，原本的索引可能不連續。drop=True 表示不保留原本的索引作為新欄位。

print("Total samples:", len(df))
print(df["Sentiment"].value_counts().sort_index())


# 2. Split dataset (70% / 15% / 15%)

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=26,
    stratify=df["Sentiment"] # 使用stratify後，Training Set和Temporary Set都會盡量維持相同的情緒類別比例。
)

# Second split: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=20,
    stratify=temp_df["Sentiment"]
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))


# RNN看不懂字，需要把love - > '15' -> embedding向量。
torch.manual_seed(14)
# Build vocabulary from training data only
counter = Counter()  # 統計每個單字出現的次數，是空的

# 為什麼只使用Training Set？
# 因為我們希望模型只利用Training Set學習。
for text in train_df["Comment"]:
    counter.update(str(text).split())  # split(): 這個動作叫做Tokenization。在這份已經預處理的資料集中，我們先用空白來分割單字。

MAX_VOCAB_SIZE = 50000

vocab = {
    "<PAD>": 0, # 用來補齊不同句子的長度
    "<UNK>": 1  # 當遇到Vocabulary沒收錄的單字時，就用 1 來表示。
}

for word, count in counter.most_common(MAX_VOCAB_SIZE - 2): # most_common()是從出現頻率最高的單字開始取出
    if word not in vocab:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))


MAX_LEN = 100  # 每句文字最多保留max_len個Token
BATCH_SIZE = 128

class SentimentDataset(Dataset):
    def __init__(self, dataframe, vocab):
        self.texts = dataframe["Comment"].astype(str).tolist()
        self.labels = dataframe["Sentiment"].astype(int).tolist()
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx].split()[:MAX_LEN] #最多到max_len

        token_ids = []
        for word in words:
            if word in self.vocab:
                token_ids.append(self.vocab[word])
            else:
                token_ids.append(self.vocab["<UNK>"])

        # 應該不會用到吧
        if len(token_ids) == 0:
            token_ids = [self.vocab["<UNK>"]]

        label = self.labels[idx]

        # 後面embedding需要long，反正要整數
        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )


# Padding(補齊句子長度)，讓不同長度的文字可以組成同一個Batch，然後送進RNN。
# collate_fn：負責決定一個 Batch 裡的資料要怎麼整理
def collate_fn(batch):
    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    # labels = (
    # tensor(2),
    # tensor(0),
    # tensor(2)
    # )


    lengths = torch.tensor(  # 紀錄每個sequences的長度，同時告訴RNN每句話真正有多長
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    padded_sequences = pad_sequence(  # 實際執行Padding
        sequences,
        batch_first=True,  #batch_first=True，表示輸出的 Tensor 形狀是：[Batch Size, Sequence Length]
        # batch_first=False，表示輸出的 Tensor 形狀是：[Sequence Length, Batch Size]
        padding_value=0
    )

    labels = torch.stack(labels) # labels原本是分開的Tensor，但我們希望把它們合併成一個Tensor。

    return padded_sequences, lengths, labels


train_dataset = SentimentDataset(train_df, vocab)
val_dataset = SentimentDataset(val_df, vocab)
test_dataset = SentimentDataset(test_df, vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# Check one batch
inputs, lengths, labels = next(iter(train_loader))

print("Input shape:", inputs.shape)
print("Lengths shape:", lengths.shape)
print("Labels shape:", labels.shape)

Device: cuda
Total samples: 240928
Sentiment
0     55105
1     82777
2    103046
Name: count, dtype: int64
Train size: 168649
Validation size: 36139
Test size: 36140
Vocabulary size: 50000
Input shape: torch.Size([128, 35])
Lengths shape: torch.Size([128])
Labels shape: torch.Size([128])


為何需要collate_fn(batch)?  
假設現在 DataLoader 要取出 3 筆資料：會發現每一句可能都不一樣長。但 lstm 通常會使用同一個 Tensor 來處理整個 Batch，因此需要把這三句話補成相同長度。  
總而言之，缺的都補0，同時lengths會跟RNN說實際的長度

In [19]:

import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(
        self,
        vocab_size,  # 字典中有多少個 Token
        embedding_dim,  # 每個單字向量有多少維
        hidden_dim,  # RNN Hidden State 的維度
        num_classes # 3個class
    ):
        super().__init__()

        # 1. Token id要轉成embedding
        # embedding這些數值不是我們人工定義的，而是模型在訓練過程中學出來的。
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0  # idx 0對應的 Embedding 保持為零向量，且不參與一般的梯度更新。
        )

        # 2. RNN Layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True  # batch_first=True是什麼？它規定輸入資料的形狀
        )

        # 3. Classification Layer
        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x, lengths):

        # x: [batch_size, seq_len]
        # 將 Token IDs 轉成 Embedding
        embedded = self.embedding(x)

        # embedded:
        # [batch_size, seq_len, embedding_dim]

        #雖然 Embedding 已經讓 Padding 對應到零向量，但一般 RNN 仍然可能繼續處理這些位置，讓 Hidden State 受到影響。
        #因此我們需要 pack_padded_sequence。
        #它會根據 lengths 整理序列，讓 RNN 知道各句的有效長度。
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False  #表示Batch裡的句子不需要事先依照長度從大到小排序。
        )

        # RNN 回傳 output 與最後的 hidden state
        packed_output, hidden = self.rnn(packed)

        # hidden: [1, batch_size, hidden_dim]
        last_hidden = hidden[-1] # 不要那個1，last_hidden = [batch_size, hidden_dim]

        # logits: [batch_size, num_classes]
        logits = self.fc(last_hidden)

        return logits  # 只是分數，還不是個類別的機率

class SentimentLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)
        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits





RNN 依序產生：
- 讀完 "I" → \(h_1\)
- 讀完 "love" → \(h_2\)
- 讀完 "this" → \(h_3\)
- 讀完 "movie" → \(h_4\)
 
因此我們會得到四個 Hidden States。  

packed_output代表h1到h4，hidden代表h4

In [20]:

EMBEDDING_DIM = 128
HIDDEN_DIM = 128

num_classes = df["Sentiment"].nunique()

rnn_model = SentimentRNN(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes
).to(device)

lstm_model = SentimentLSTM(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()

rnn_optimizer = torch.optim.Adam(
    rnn_model.parameters(),
    lr=0.001
)

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)



In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score

def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    all_predictions = []
    all_labels = []

    for inputs, lengths, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_training):
            logits = model(inputs, lengths)
            loss = criterion(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        # 因為一個batch有128個，predictions = [......]有128個，而append會把整個list當作一個element加入
        # extend會把list裡面的每個元素分別加入
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    precision = precision_score(all_labels, all_predictions, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_predictions, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy, precision, recall, f1


In [22]:

import copy

EPOCHS = 20
rnn_best_val_loss = float("inf")
rnn_best_model_state = None

print(f"RNN model: ")
for epoch in range(EPOCHS):

    train_loss, train_acc, train_precision, train_recall, train_f1 = run_epoch(
        rnn_model, train_loader, criterion, rnn_optimizer
    )

    val_loss, val_acc, val_precision, val_recall, val_f1 = run_epoch(
        rnn_model, val_loader, criterion
    )

    print(
        f"Train | Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"Precision: {train_precision:.4f} | "
        f"Recall: {train_recall:.4f} | "
        f"F1: {train_f1:.4f}"
    )

    print(
        f"Val   | Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"Precision: {val_precision:.4f} | "
        f"Recall: {val_recall:.4f} | "
        f"F1: {val_f1:.4f}"
    )

    if val_loss < rnn_best_val_loss:
        rnn_best_val_loss = val_loss
        rnn_best_model_state = copy.deepcopy(rnn_model.state_dict())

rnn_model.load_state_dict(rnn_best_model_state)


RNN model: 
Train | Loss: 0.8567 | Acc: 0.6167 | Precision: 0.5990 | Recall: 0.5746 | F1: 0.5772
Val   | Loss: 0.7067 | Acc: 0.7216 | Precision: 0.7091 | Recall: 0.7091 | F1: 0.7072
Train | Loss: 0.6107 | Acc: 0.7704 | Precision: 0.7596 | Recall: 0.7514 | F1: 0.7547
Val   | Loss: 0.6088 | Acc: 0.7787 | Precision: 0.7706 | Recall: 0.7615 | F1: 0.7654
Train | Loss: 0.5119 | Acc: 0.8131 | Precision: 0.8049 | Recall: 0.7988 | F1: 0.8015
Val   | Loss: 0.5917 | Acc: 0.7829 | Precision: 0.7741 | Recall: 0.7666 | F1: 0.7699
Train | Loss: 0.4711 | Acc: 0.8291 | Precision: 0.8211 | Recall: 0.8176 | F1: 0.8193
Val   | Loss: 0.5474 | Acc: 0.8038 | Precision: 0.7933 | Recall: 0.7925 | F1: 0.7929
Train | Loss: 0.4226 | Acc: 0.8467 | Precision: 0.8388 | Recall: 0.8361 | F1: 0.8374
Val   | Loss: 0.5466 | Acc: 0.8074 | Precision: 0.8010 | Recall: 0.7901 | F1: 0.7947
Train | Loss: 0.3707 | Acc: 0.8677 | Precision: 0.8607 | Recall: 0.8586 | F1: 0.8596
Val   | Loss: 0.5533 | Acc: 0.8090 | Precision: 0.800

<All keys matched successfully>

In [23]:
EPOCHS = 20
lstm_best_val_loss = float("inf")
lstm_best_model_state = None

for epoch in range(EPOCHS):

    train_loss, train_acc, train_precision, train_recall, train_f1 = run_epoch(
        lstm_model, train_loader, criterion, lstm_optimizer
    )

    val_loss, val_acc, val_precision, val_recall, val_f1 = run_epoch(
        lstm_model, val_loader, criterion
    )

    print(f"LSTM model: ")
    print(
    f"Train | Loss: {train_loss:.4f} | "
    f"Acc: {train_acc:.4f} | "
    f"Precision: {train_precision:.4f} | "
    f"Recall: {train_recall:.4f} | "
    f"F1: {train_f1:.4f}"
    )

    print(
        f"Val   | Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"Precision: {val_precision:.4f} | "
        f"Recall: {val_recall:.4f} | "
        f"F1: {val_f1:.4f}"
    )

    if val_loss < lstm_best_val_loss:
        lstm_best_val_loss = val_loss
        lstm_best_model_state = copy.deepcopy(lstm_model.state_dict())

lstm_model.load_state_dict(lstm_best_model_state)


LSTM model: 
Train | Loss: 0.7352 | Acc: 0.6888 | Precision: 0.6801 | Recall: 0.6549 | F1: 0.6607
Val   | Loss: 0.5863 | Acc: 0.7728 | Precision: 0.7671 | Recall: 0.7515 | F1: 0.7572
LSTM model: 
Train | Loss: 0.4898 | Acc: 0.8149 | Precision: 0.8081 | Recall: 0.8002 | F1: 0.8032
Val   | Loss: 0.5031 | Acc: 0.8096 | Precision: 0.8009 | Recall: 0.8002 | F1: 0.7997
LSTM model: 
Train | Loss: 0.3874 | Acc: 0.8573 | Precision: 0.8512 | Recall: 0.8471 | F1: 0.8487
Val   | Loss: 0.4890 | Acc: 0.8199 | Precision: 0.8184 | Recall: 0.8013 | F1: 0.8075
LSTM model: 
Train | Loss: 0.3050 | Acc: 0.8914 | Precision: 0.8863 | Recall: 0.8837 | F1: 0.8848
Val   | Loss: 0.4908 | Acc: 0.8289 | Precision: 0.8206 | Recall: 0.8193 | F1: 0.8197
LSTM model: 
Train | Loss: 0.2328 | Acc: 0.9192 | Precision: 0.9149 | Recall: 0.9138 | F1: 0.9143
Val   | Loss: 0.5263 | Acc: 0.8284 | Precision: 0.8173 | Recall: 0.8236 | F1: 0.8201
LSTM model: 
Train | Loss: 0.1684 | Acc: 0.9435 | Precision: 0.9404 | Recall: 0.9397 

<All keys matched successfully>

In [24]:

test_loss, test_acc, test_precision, test_recall, test_f1 = run_epoch(
    rnn_model, test_loader, criterion
)

print("RNN test: ")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")


test_loss, test_acc, test_precision, test_recall, test_f1 = run_epoch(
    lstm_model, test_loader, criterion
)

print("LSTM test: ")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")



RNN test: 
Test Loss: 0.5336
Test Accuracy: 0.8126
Test Precision: 0.8072
Test Recall: 0.7950
Test F1: 0.8001
LSTM test: 
Test Loss: 0.4832
Test Accuracy: 0.8232
Test Precision: 0.8215
Test Recall: 0.8045
Test F1: 0.8109
